In [1]:
# 07_pareto_all_adversaries.ipynb -- Pareto frontier drawn for ALL three adversary strengths, replacing the single mid-adversary figure.
# Makes explicit that for the strong adversary C2 and C3 are a trade-off (not domination). No LLM calls.
# Output paths are intentionally not printed (anonymity).

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 120, "font.size": 11,
                     "axes.edgecolor": "0.2", "grid.color": "0.85", "image.cmap": "Greys"})

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
TAB, FIG = ROOT / "results" / "tables", ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

risk = pd.read_csv(TAB / "risk_by_tier.csv")
util = pd.read_csv(TAB / "utility_by_tier.csv")
df = risk.merge(util, on="tier")
TIERS = ["C0", "C1", "C2", "C3", "C4"]
df["tier"] = pd.Categorical(df["tier"], TIERS, ordered=True)
df = df.sort_values("tier")

def nondominated(points):
    # points: list of (tier, risk, utility). Non-dominated = no other point has <= risk AND >= utility (strictly better somewhere).
    nd = []
    for t, x, y in points:
        dominated = any((xx <= x and yy >= y) and (xx < x or yy > y) for tt, xx, yy in points if tt != t)
        if not dominated:
            nd.append((t, x, y))
    return nd

OFF = {"C0": (8, 6), "C1": (8, -14), "C2": (9, 6), "C3": (-26, 8), "C4": (9, -14)}

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.6), sharey=True)
for ax, model in zip(axes, ["weak", "mid", "strong"]):
    pts = [(r["tier"], r[f"risk_{model}"], r["task_success_proxy"]) for _, r in df.iterrows()]
    nd = {t for t, _, _ in nondominated([(t, x, y) for t, x, y in pts if t != "C0"])}
    for t, x, y in pts:
        compliant = t != "C0"
        face = "0.25" if compliant else "white"
        ax.scatter(x, y, s=70, facecolors=face, edgecolors="0.1", linewidths=1.3, zorder=3)
        dx, dy = OFF[t]
        ax.annotate(t, (x, y), textcoords="offset points", xytext=(dx, dy), fontsize=9)
    # connect non-dominated compliant tiers
    ndpts = sorted([(x, y) for t, x, y in pts if t in nd], key=lambda p: p[0])
    if len(ndpts) >= 2:
        ax.plot([p[0] for p in ndpts], [p[1] for p in ndpts], color="0.1", linewidth=1.2, zorder=2)
    ax.set_title(f"{model}", fontsize=10)
    ax.set_xlabel("Reconstruction risk")
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.03, 1.03)
axes[0].set_ylabel("Utility (task-success proxy)")
save_fig = lambda n: (fig.savefig(FIG / f"{n}.png", dpi=600, bbox_inches="tight"),
                      fig.savefig(FIG / f"{n}.pdf", bbox_inches="tight"), plt.close(fig))
save_fig("fig_pareto_all_adversaries")

# Report the non-dominated set per adversary.
rows = []
for model in ["weak", "mid", "strong"]:
    pts = [(r["tier"], r[f"risk_{model}"], r["task_success_proxy"]) for _, r in df.iterrows() if r["tier"] != "C0"]
    nd = [t for t, _, _ in nondominated(pts)]
    rows.append({"model": model, "non_dominated_tiers": ",".join(map(str, nd))})
out = pd.DataFrame(rows)
out.to_csv(TAB / "pareto_nondominated.csv", index=False)
print("Non-dominated compliant tiers per adversary:")
print(out.to_string(index=False))


Non-dominated compliant tiers per adversary:
 model non_dominated_tiers
  weak               C1,C2
   mid         C1,C2,C3,C4
strong         C1,C2,C3,C4
